# SOSSA Profiling

Profiles the SOSSA (sum-of-squares spectrum amplification) algorithm for Fe2S2 using:
* The Q# interpreter (`use_qir=False`).
* `SequentialCountingProfiler`, which counts instructions and does nothing else.

Profiling with the Q# interpreter is very slow (10,000 ns/op), and we expect profiling with QIR to take 40 ns/op with PyQIR and the bytecode runner, and 10 ns/op with JIT compilation.

We reduced the `num_queries` parameter from 10,162 to 20. Profiling time scales linearly, so we expect profiling the original algorithm to take about 4 hours.

The code is based on [this notebook](https://github.com/microsoft/qdk-chemistry/blob/main/examples/sossa_qre.ipynb). To reproduce, you need to install qdk-chemistry: `pip install qdk-chemistry==2.3.0`.


In [1]:
import math

import numpy as np
from qdk.qre.application import QSharpApplication
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef,
    Configuration,
    DFTHCHamiltonianContainer,
    Hamiltonian,
    MajoranaMapping,
    ModelOrbitals,
    StateVectorContainer,
    Wavefunction,
)


def get_sossa_app(num_queries_override: int = None) -> QSharpApplication:
    FE2S2_PARAMETERS = {
        "electrons": 30,
        "N": 20,
        "R": 14,
        "B": 15,
        "C": 5,
        "lambda_eff": 6.4690,
        "b_coeff": 11,
        "b_rot": 15,
    }
    TARGET_PRECISION = 1e-3
    seed = 42
    fe2s2_num_queries = math.ceil(
        math.pi * FE2S2_PARAMETERS["lambda_eff"] / (2 * TARGET_PRECISION)
    )

    if num_queries_override is not None:
        print(f"Changed num_queries from {fe2s2_num_queries} to {num_queries_override}")
        fe2s2_num_queries = num_queries_override

    n, r, b, c = (FE2S2_PARAMETERS[key] for key in ("N", "R", "B", "C"))
    rng = np.random.default_rng(seed)
    one_body = rng.standard_normal((n, n))
    basis_vectors = rng.standard_normal((r, b, n))
    basis_vectors /= np.linalg.norm(basis_vectors, axis=-1, keepdims=True)
    synthetic_fe2_s2_hamiltonian = Hamiltonian(
        DFTHCHamiltonianContainer(
            one_body_integrals=(one_body + one_body.T) / 2,
            u_matrices=basis_vectors.ravel(),
            w_matrices=0.1 * rng.standard_normal((r, b, c)).ravel(),
            wb_matrix=0.1 * rng.standard_normal((r, c)),
            orbitals=ModelOrbitals(n),
            core_energy=0.0,
            inactive_fock_matrix=np.zeros((n, n)),
        )
    )
    fe2s2_operator = create("qubit_mapper", "sum_of_squares").run(
        synthetic_fe2_s2_hamiltonian, MajoranaMapping.jordan_wigner(2 * n)
    )

    container = synthetic_fe2_s2_hamiltonian.get_container()
    configuration = Configuration.canonical_hf_configuration(
        (FE2S2_PARAMETERS["electrons"] + 1) // 2,
        FE2S2_PARAMETERS["electrons"] // 2,
        container.get_num_orbitals(),
    )
    reference = Wavefunction(
        StateVectorContainer(configuration, container.get_orbitals())
    )
    fe2s2_state_preparation = create("state_prep", "sparse_isometry").run(reference)

    sossa_unitary_builder = AlgorithmRef("hamiltonian_unitary_builder", "sossa")
    fe2s2_circuit = create(
        "qpe_circuit_builder",
        "qdk_unary",
        num_queries=fe2s2_num_queries,
        circuit_mapper=AlgorithmRef(
            "circuit_mapper",
            "sossa",
            outer_prepare_algorithm=AlgorithmRef("state_prep", "alias_sampling"),
            inner_prepare_algorithm="controlled_alias_sampling",
            select_algorithm="qrom_phase_gradient",
            rotation_bit_precision=FE2S2_PARAMETERS["b_rot"],
            coefficient_bit_precision=FE2S2_PARAMETERS["b_coeff"],
        ),
        unitary_builder=sossa_unitary_builder,
    ).run(
        state_preparation=fe2s2_state_preparation,
        qubit_hamiltonian=fe2s2_operator,
    )[
        0
    ]
    return fe2s2_circuit.get_qre_application()

In [2]:
from time import perf_counter

from qdk.qre.profiler import QREProfiler

def report_result(result, t0):
    dt = perf_counter() - t0
    print("Logical qubits:", result.totals.logical_qubits)
    print("Total operations:", result.totals.total_operations_count)
    print("Operation counts:", result.totals.calls_per_operation)
    per_op = f"{int(1e9 * dt / result.totals.total_operations_count)} ns/op"
    print(f"Profiling time {dt:.2f}s, {per_op}")
    

app = get_sossa_app(num_queries_override=20)

profiler = QREProfiler.counting()
t0 = perf_counter()
result = profiler.profile(app, use_qir=False)
report_result(result, t0)

Changed num_queries from 10162 to 20
[2026-11-12 12:16:30.771859] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] After filtering, only 1 determinant remains, using single reference state preparation
Logical qubits: 468
Total operations: 2707559
Operation counts: {'cx': 2161363, 'rz': 691, 'ccx': 83813, 'z': 3137, 'h': 103098, 'x': 52184, 'sadj': 3045, 'mz_store_reset': 94618, 'read_result': 94618, 'cz': 107757, 'reset': 190, 's': 3045}
Profiling time 27.29s, 10078 ns/op
